# Part 3 - Fine-tune Gemma on a TPU/GPU with LoRA (Keras 3, JAX backend)

Follows Google's tutorial *Fine-tune Gemma in Keras using LoRA* (KerasHub `Gemma3CausalLM`, preset `gemma3_instruct_1b`).

**Run this notebook unchanged once per accelerator**: first **T4 GPU**, then **TPU v5e-1**
(`Runtime -> Change runtime type`). For each run record, from the last cells:
`device`, `epoch_s`, `peak_mem_GB`, the two generations **before** and **after** fine-tuning,
and the `keras` / `keras_hub` versions.

## Prerequisites (do once)
1. Accept the Gemma license at https://www.kaggle.com/models/google/gemma with your Kaggle account.
2. Kaggle -> Settings -> API -> **Create New Token** (downloads `kaggle.json`).
3. In Colab, left panel -> key icon (**Secrets**) -> add `KAGGLE_USERNAME` and `KAGGLE_KEY`,
   and enable notebook access for both.

In [ ]:
# 1) Backend, Kaggle secrets, and install WITHOUT moving JAX.
# NEVER pip install -U jax on Colab: it breaks the CUDA (GPU) / libtpu (TPU) plugin pairing.
import os, jax, subprocess, sys
os.environ["KERAS_BACKEND"] = "jax"                 # Keras 3 on the JAX backend
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.0"  # let the model use the whole device

from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

open("c.txt", "w").write(f"jax=={jax.__version__}\n")   # pin JAX exactly as Colab ships it
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-c", "c.txt", "-U", "keras", "keras-hub"], check=True)

import keras, keras_hub
print("keras", keras.__version__, "| keras_hub", keras_hub.__version__,
      "| jax", jax.__version__, "| device", jax.devices()[0].device_kind)

In [ ]:
# 2) Dataset: Databricks Dolly 15k (same as the tutorial). Keep a 1000-example subset.
import json
!wget -q -O dolly.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

data = []
with open("dolly.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["context"]:                 # use only examples without extra context
            continue
        data.append("Instruction:\n{instruction}\n\nResponse:\n{response}".format(**ex))
data = data[:1000]
print(len(data), "training examples")
print(data[0][:300])

In [ ]:
# 3) Load Gemma and generate BEFORE fine-tuning.
keras.config.set_dtype_policy("bfloat16")           # fit the model in memory
gemma = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma.summary()

PROMPTS = [
    "Instruction:\nExplain in one sentence what a TPU is.\n\nResponse:\n",
    "Instruction:\nWhy is bfloat16 used to train neural networks?\n\nResponse:\n",
]
print("===== BEFORE fine-tuning =====")
before = [gemma.generate(p, max_length=128) for p in PROMPTS]
for b in before:
    print(b, "\n" + "-" * 70)

In [ ]:
# 4) Enable LoRA, fine-tune one epoch, and measure time + peak memory.
import time
gemma.backbone.enable_lora(rank=4)                  # LoRA, rank 4
gemma.preprocessor.sequence_length = 256            # reduce if you hit OOM (apply on BOTH devices)

opt = keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=0.01)
opt.exclude_from_weight_decay(var_names=["bias", "scale"])
gemma.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=opt,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

t0 = time.time()
gemma.fit(data, epochs=1, batch_size=1)
epoch_s = time.time() - t0

try:
    peak_gb = jax.local_devices()[0].memory_stats().get("peak_bytes_in_use", 0) / 1e9
except Exception as e:
    peak_gb = float("nan"); print("memory_stats unavailable:", e)

DEVICE = jax.devices()[0].device_kind
print(f"\ndevice={DEVICE}  epoch_s={epoch_s:.1f}  peak_mem_GB={peak_gb:.2f}")

In [ ]:
# 5) Generate AFTER fine-tuning and save a one-line summary for the report.
print("===== AFTER fine-tuning =====")
after = [gemma.generate(p, max_length=128) for p in PROMPTS]
for a in after:
    print(a, "\n" + "-" * 70)

summary = {
    "device": DEVICE, "epoch_s": round(epoch_s, 1), "peak_mem_GB": round(peak_gb, 2),
    "keras": keras.__version__, "keras_hub": keras_hub.__version__, "jax": jax.__version__,
}
print("\nRESULT LINE (copy this into the report):")
print(summary)

with open("gemma_part3_results.txt", "w") as f:
    f.write(str(summary) + "\n\n== BEFORE ==\n" + "\n---\n".join(before)
            + "\n\n== AFTER ==\n" + "\n---\n".join(after))
try:
    from google.colab import files
    files.download("gemma_part3_results.txt")
except Exception:
    pass